# 35. Attention, self-attention, and causal masking

**Level:** Intermediate to advanced

**Prerequisites:** 07, 10, and 27: matrix operations, probabilities, and neural concepts.

[Course index](00_START_HERE.ipynb) | [Previous](34_Embeddings_Similarity_and_Named_Entities.ipynb) | [Next](36_Transformer_Blocks_and_Positions.ipynb)

**Topics:** Attention, Self-attention, Queries, Keys, Values, Causal mask, Multi-head attention.

**Matching script:** `examples\35_Attention_and_Causal_Masking.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Attention forms a weighted combination of value vectors. A query compares with keys to produce
scores, softmax turns scores into weights, and weights combine values. In self-attention, the
queries, keys, and values are projected from the same input sequence.

A causal language model must not use future target tokens when predicting a position. An
upper-triangular mask sets future-position scores to negative infinity before softmax.
Padding masks solve a different problem: they exclude artificial padding positions.

Multi-head attention uses several learned projection subspaces, then combines their outputs.
A head is not guaranteed to specialize in one human-interpretable linguistic rule. Attention
weights reveal one computational mechanism, not a complete faithful explanation of a model.

The original transformer paper supplies the architectural foundation; current architectures
vary in normalization, positional representations, head sharing, sparsity, and other details.

## Visual guide

![Attention, self-attention, and causal masking teaching diagram](images/attention.png)

Trace the diagram left to right and identify the input, transformation, and output. It illustrates the mechanism, not measured model quality.

## Mathematics or data contract

Attention(Q,K,V)=softmax(QK^T/sqrt(d_key) + mask)V, with softmax applied along key positions.

## Important limitation

Identity projections on a three-token sequence demonstrate the calculation, not learned language behavior.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Compute masked scaled dot-product attention

**Purpose:** Verify both the math and the no-future-information invariant.

### Read the code in this order
1. Create Q, K, V from one small sequence.
2. Compute scaled scores.
3. Mask future key positions before stable softmax.
4. Check row sums and exact zero future weights.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
attention weights: [[1.0, 0.0, 0.0], [0.33, 0.67, 0.0], [0.248, 0.248, 0.503]]
output shape: (3, 2)
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

X = np.array([[1., 0.], [0., 1.], [1., 1.]])
Q, K, V = X.copy(), X.copy(), X.copy()
scores = Q @ K.T / np.sqrt(K.shape[1])
future = np.triu(np.ones((3, 3), dtype=bool), k=1)
scores[future] = -np.inf
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)
output = weights @ V
np.testing.assert_allclose(weights.sum(axis=1), 1)
assert np.all(weights[future] == 0)
np.testing.assert_allclose(output[0], V[0])
print("attention weights:", np.round(weights, 3).tolist())
print("output shape:", output.shape)

attention weights: [[1.0, 0.0, 0.0], [0.33, 0.67, 0.0], [0.248, 0.248, 0.503]]
output shape: (3, 2)


### Interpret example 1

The result should demonstrate this purpose: **Verify both the math and the no-future-information invariant.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Change the final token's value and prove the first output stays unchanged under causal masking.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Attention` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Attention(Q,K,V)=softmax(QK^T/sqrt(d_key) + mask)V, with softmax applied along key positions.
- Name the boundary of the demonstration: Identity projections on a three-token sequence demonstrate the calculation, not learned language behavior.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/1706.03762`
- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.